In [2]:
! pip install geopandas

> 필요한 모듈 불러오기

In [13]:
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.stats.mstats import gmean
from shapely.geometry import Point

import seaborn as sns
import matplotlib.pyplot as plt

> 파일 불러오기

In [ ]:
# 파일 경로 설정 (.shp 확장자를 가진 파일을 선택하세요)
# 나머지 .dbf, .shx, .prj 파일들은 같은 폴더 내에 있어야 자동으로 함께 읽힙니다.
file_path = './LSMD_ADM_SECT_UMD_제주/' + 'LSMD_ADM_SECT_UMD_50_202512.shp'

#파일 불러오기
#data = gpd.read_file(file_path)
one = gpd.read_file(file_path, encoding='cp949')
one.to_csv('LSMD_ADM_SECT_UMD_제주_202512.csv', encoding='cp949', mode='w', index=True)
one

,EMD_CD,COL_ADM_SE,EMD_NM,SGG_OID,geometry
0,50110253,50110,애월읍,1821,"MULTIPOLYGON (((134706.675 94991.383, 134716.4..."
1,50110259,50110,조천읍,1813,"MULTIPOLYGON (((159001.754 87069.627, 159012.5..."
2,50110113,50110,삼양일동,1797,"POLYGON ((161553.644 103603.479, 161561.553 10..."
3,50110104,50110,이도이동,1793,"POLYGON ((155924.582 100821.095, 155924.976 10..."
4,50110105,50110,삼도일동,1791,"POLYGON ((154889.957 101422.409, 154899.182 10..."
...,...,...,...,...,...
69,50130102,50130,법환동,418,"MULTIPOLYGON (((155545.022 72083.341, 155541.8..."
70,50130103,50130,서호동,417,"POLYGON ((154372.11 73875.46, 154376.786 74212..."
71,50130108,50130,하효동,375,"POLYGON ((163575.489 73099.714, 163576.44 7310..."
72,50130259,50130,성산읍,2322,"MULTIPOLYGON (((181418.747 87642.923, 181426.6..."


In [21]:
file_path2 = './제주도_지형_정보/' + '제주도_지형_정보.shp'

two = gpd.read_file(file_path2, encoding='cp949')
two = two.to_crs(one.crs)
joined_onetwo = gpd.sjoin(two, one, how='left', predicate='within')
target_counts = joined_onetwo.groupby('EMD_NM').size().reset_index(name='count')
# joined_onetwo
# target_counts
joined_onetwo.to_csv('제주도_지형_정보2.csv', encoding='cp949', mode='w', index=True)

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: ./제주도_지형_정보/제주도_지형_정보.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


##### EMD_CD
 - EupMyeonDong Code 읍면동 코드
 - (법정동 또는 행정동을 식별하는 고유 번호입니다.)
##### COL_ADM_SE
 - Collection Administrative Section 행정 구역 구분
 - (데이터 수집 및 관리 기준이 되는 행정 구역 구분 코드입니다.)
##### EMD_NM
 - EupMyeonDong Name 읍면동 명칭 (예: '이도이동', '성산읍'과 같은 실제 지역 이름입니다.)
##### SGG_OID
 - SiGunGu Object ID 시군구 관리 번호
 - (해당 데이터 객체의 시스템 관리용 고유 ID입니다.)
##### geometry
 - Geometry 지형 정보
 - (해당 지역의 경계선을 그리는 좌표들의 집합(Polygon)입니다. data.plot()을 실행할 때 이 데이터를 기반으로 지도가 그려집니다.)

> 요소별 가중치 작성

- 기하평균

- 쌍대비교행렬

- 정규화

In [ ]:
# 1. 데이터 준비 (각 행은 지역, 열은 평가항목 점수)
# 예: 5개 지역에 대한 6개 지표 데이터
data = np.array([
    [0.8, 0.7, 0.2, 0.9, 0.6, 0.5], # 지역 1
    [0.6, 0.9, 0.1, 0.8, 0.5, 0.4], # 지역 2
    # ... (보유하신 GIS/API 데이터를 기반으로 수치화)
])

# 2. 정규화 (Normalization)
P = data / data.sum(axis=0)

# 3. 엔트로피 계산
k = 1 / np.log(len(data))
E = -k * (P * np.log(P + 1e-10)).sum(axis=0)

# 4. 가중치 도출
d = 1 - E
weights = d / d.sum()

print("도출된 항목별 가중치:", weights)

In [ ]:
def AHP_6(a12, a13, a14, a15, a16, a23, a24, a25, a26, a34, a35, a36, a45, a46, a56):
    # 6x6 쌍대비교 행렬 생성
    matrix = np.array([
        [1, a12, a13, a14, a15, a16],
        [1/a12, 1, a23, a24, a25, a26],
        [1/a13, 1/a23, 1, a34, a35, a36],
        [1/a14, 1/a24, 1/a34, 1, a45, a46],
        [1/a15, 1/a25, 1/a35, 1/a45, 1, a56],
        [1/a16, 1/a26, 1/a36, 1/a46, 1/a56, 1]
    ])
    
    # 가중치 계산 (정규화 및 행 평균)
    col_sums = matrix.sum(axis=0)
    norm_matrix = matrix / col_sums
    weights = norm_matrix.mean(axis=1)
    
    # 일관성 지수(CI) 계산
    # n=6 일 때 RI 값은 보통 1.24를 사용합니다.
    weighted_sum = matrix.dot(weights)
    consistency_val = weighted_sum / weights
    ci = (consistency_val.mean() - 6) / 5
    cr = ci / 1.24  # 일관성 비율
    
    return weights, cr

> 일관성 지수 도출


  - 쌍대비교행렬과 가중치 행렬을 곱한다.
  - 곱한 행렬의 각 값을 가중치로 나눈다.
  - 각 값의 평균에서 요소의 개수(여기서는 3)을 빼고 요소의 개수보다 1작은 수(여기서는 2)로 나눈다.

In [ ]:
# 이미지의 가중치를 역산하거나 본인이 설정한 쌍대비교 행렬(6x6)
# 예시 행렬 (접근성, 보존, 이용객, 안전, 개발, 통합)
data = [
    [1, 2.5, 1, 1.25, 2.5, 2.5],
    [0.4, 1, 0.4, 0.5, 1, 1],
    # ... 나머지 행 생략
]
labels = ['Access', 'Conserve', 'Traveler', 'Safety', 'Exploit', 'Integrate']

plt.figure(figsize=(8, 6))
sns.heatmap(data, annot=True, xticklabels=labels, yticklabels=labels, cmap='YlGnBu')
plt.title("AHP Pairwise Comparison Matrix")
plt.show()